In [0]:
#=============================================================================
# SCD TYPE 2 using SNAPSHOT_ID (current_timestamp) as part of the Surrogate Key
# =============================================================================
# 
# CONCEPT OVERVIEW:
# -----------------
# In SCD Type 2, we keep FULL HISTORY of every change to a record.
# Each time a record changes, we:
#   1. EXPIRE the old row (set end_date, is_current = False)
#   2. INSERT a new row with the updated values (is_current = True)
#
# SNAPSHOT_ID:
# - A timestamp captured at the SOURCE at the time of extraction.
# - Think of it as: "When was this data snapshot taken?"
# - We use current_timestamp() to simulate this.
#
# SURROGATE KEY:
# - A unique identifier for each VERSION of a record.
# - Built by hashing: business_key (id) + snapshot_id
# - This guarantees that EACH VERSION of the same customer gets a DIFFERENT key.
#   e.g., Customer 101 on Jan 1 → SK "abc123"
#         Customer 101 on Jun 15 (after change) → SK "xyz789"
#
# WHY hash(id + snapshot_id)?
# - The business key (id) alone can't be unique — the same customer has
#   MULTIPLE rows in SCD Type 2 (one per version).
# - Adding snapshot_id makes it unique per version.
# =============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, sha2, concat_ws, when
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)
from datetime import datetime

# =====================
# STEP 1: SOURCE DATA — "Day 1" Initial Snapshot
# =====================
# Imagine your source system takes a snapshot at 2024-01-01 08:00:00.
# The snapshot_id = the timestamp when the source extracted the data.

day1_snapshot = datetime(2024, 1, 1, 8, 0, 0)

schema = StructType([
    StructField("id", IntegerType(), False),          # Business Key
    StructField("name", StringType(), False),
    StructField("city", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("snapshot_id", TimestampType(), False) # When source extracted this data
])

initial_data = [
    (101, "Alice",  "Seattle",  70000.0, day1_snapshot),
    (102, "Bob",    "Portland", 85000.0, day1_snapshot),
    (103, "Carol",  "Denver",   60000.0, day1_snapshot),
]

df_day1 = spark.createDataFrame(initial_data, schema=schema)
print("📸 SOURCE SNAPSHOT — Day 1 (2024-01-01 08:00)")
print("   snapshot_id = timestamp when source system extracted this batch")
df_day1.show(truncate=False)

# =====================
# STEP 2: BUILD SCD TYPE 2 TARGET — Day 1 (Initial Load)
# =====================
# For each row, we create:
#   - surrogate_key = SHA-256 hash of (id + snapshot_id) → unique per version
#   - effective_date = snapshot_id (when this version became active)
#   - end_date = NULL (still current)
#   - is_current = True

df_scd2_day1 = df_day1.withColumn(
    "surrogate_key",
    sha2(concat_ws("|", col("id").cast("string"), col("snapshot_id").cast("string")), 256)
).withColumn(
    "effective_date", col("snapshot_id")
).withColumn(
    "end_date", lit(None).cast("timestamp")
).withColumn(
    "is_current", lit(True)
).select(
    "surrogate_key", "id", "name", "city", "salary",
    "snapshot_id", "effective_date", "end_date", "is_current"
)

print("✅ SCD TYPE 2 TARGET after Day 1 Initial Load")
print("   surrogate_key = SHA-256(id | snapshot_id) — unique per version!")
df_scd2_day1.show(truncate=False)

# =====================
# STEP 3: SOURCE DATA — "Day 2" New Snapshot (with changes!)
# =====================
# On June 15, the source system takes another snapshot.
# Changes:
#   - id=101 (Alice): moved to "San Francisco" and salary raised to 90000
#   - id=103 (Carol): salary raised to 75000
#   - id=104 (Dave):  brand-new employee
#   - id=102 (Bob):   NO CHANGE

day2_snapshot = datetime(2024, 6, 15, 10, 30, 0)

update_data = [
    (101, "Alice",  "San Francisco", 90000.0, day2_snapshot),  # CHANGED city & salary
    (102, "Bob",    "Portland",      85000.0, day2_snapshot),  # UNCHANGED
    (103, "Carol",  "Denver",        75000.0, day2_snapshot),  # CHANGED salary
    (104, "Dave",   "Austin",        55000.0, day2_snapshot),  # NEW
]

df_day2 = spark.createDataFrame(update_data, schema=schema)
print("📸 SOURCE SNAPSHOT — Day 2 (2024-06-15 10:30)")
print("   Alice changed city+salary, Carol changed salary, Dave is NEW, Bob unchanged")
df_day2.show(truncate=False)

# =====================
# STEP 4: DETECT CHANGES (compare Day 2 source vs current SCD2 target)
# =====================
# Join on business key (id) to find:
#   - CHANGED rows: same id, different values
#   - NEW rows: id exists in source but not in target
#   - UNCHANGED rows: same id, same values

tracked_cols = ["name", "city", "salary"]

df_joined = df_scd2_day1.filter(col("is_current") == True).alias("target").join(
    df_day2.alias("source"),
    col("target.id") == col("source.id"),
    "full_outer"
)

# Build change detection condition
change_condition = None
for c in tracked_cols:
    cond = (
        (col(f"target.{c}") != col(f"source.{c}")) |
        (col(f"target.{c}").isNull() & col(f"source.{c}").isNotNull()) |
        (col(f"target.{c}").isNotNull() & col(f"source.{c}").isNull())
    )
    change_condition = cond if change_condition is None else (change_condition | cond)

# --- CHANGED records (exist in both, values differ) ---
df_changed = df_joined.filter(
    col("target.id").isNotNull() & col("source.id").isNotNull() & change_condition
)

# --- NEW records (exist in source only) ---
df_new = df_joined.filter(col("target.id").isNull())

# --- UNCHANGED records (exist in both, no change) ---
df_unchanged = df_joined.filter(
    col("target.id").isNotNull() & col("source.id").isNotNull() & ~change_condition
)

print(f"🔍 Change Detection Results:")
print(f"   CHANGED:   {df_changed.count()} rows (will get expired + new version)")
print(f"   NEW:       {df_new.count()} rows (will be inserted)")
print(f"   UNCHANGED: {df_unchanged.count()} rows (stay as-is)")

# =====================
# STEP 5: BUILD THE FINAL SCD TYPE 2 TABLE
# =====================

# 5a. EXPIRE old versions of changed records
df_expired = df_changed.select(
    col("target.surrogate_key"),
    col("target.id"),
    col("target.name"),
    col("target.city"),
    col("target.salary"),
    col("target.snapshot_id"),
    col("target.effective_date"),
    col("source.snapshot_id").alias("end_date"),   # Expired AT the new snapshot time
    lit(False).alias("is_current")
)


# 5b. NEW VERSIONS of changed records (with new surrogate key!)
df_new_versions = df_changed.select(
    sha2(concat_ws("|", col("source.id").cast("string"), col("source.snapshot_id").cast("string")), 256).alias("surrogate_key"),
    col("source.id"),
    col("source.name"),
    col("source.city"),
    col("source.salary"),
    col("source.snapshot_id"),
    col("source.snapshot_id").alias("effective_date"),
    lit(None).cast("timestamp").alias("end_date"),
    lit(True).alias("is_current")
)

# 5c. UNCHANGED rows (keep as-is from target)
df_unchanged_rows = df_unchanged.select(
    col("target.surrogate_key"),
    col("target.id"),
    col("target.name"),
    col("target.city"),
    col("target.salary"),
    col("target.snapshot_id"),
    col("target.effective_date"),
    col("target.end_date"),
    col("target.is_current")
)

# 5d. BRAND-NEW customers
df_new_customers = df_new.select(
    sha2(concat_ws("|", col("source.id").cast("string"), col("source.snapshot_id").cast("string")), 256).alias("surrogate_key"),
    col("source.id"),
    col("source.name"),
    col("source.city"),
    col("source.salary"),
    col("source.snapshot_id"),
    col("source.snapshot_id").alias("effective_date"),
    lit(None).cast("timestamp").alias("end_date"),
    lit(True).alias("is_current")
)

# 5e. UNION ALL parts together
df_scd2_final = (
    df_expired
    .unionByName(df_new_versions)
    .unionByName(df_unchanged_rows)
    .unionByName(df_new_customers)
    .orderBy("id", "effective_date")
)

print("\n" + "="*90)
print("🏆 FINAL SCD TYPE 2 TABLE (after Day 2 processing)")
print("="*90)
print("""
KEY OBSERVATIONS:
─────────────────
* id=101 (Alice): 2 rows — old version EXPIRED (is_current=False, end_date set),
                            new version with San Francisco + 90k (is_current=True)
* id=102 (Bob):   1 row  — UNCHANGED, still current
* id=103 (Carol): 2 rows — old version EXPIRED, new version with salary 75k
* id=104 (Dave):  1 row  — BRAND NEW employee

SURROGATE KEY logic:
────────────────────
* surrogate_key = SHA-256(id | snapshot_id)
* Alice Day 1: SHA-256("101|2024-01-01 08:00:00")  → one hash
* Alice Day 2: SHA-256("101|2024-06-15 10:30:00")  → DIFFERENT hash!
* This guarantees each VERSION of a record has a UNIQUE surrogate key.
""")
df_scd2_final.show(truncate=False)

# Save to Delta
df_scd2_final.write.format("delta").mode("overwrite").saveAsTable("employees_scd2_snapshot")
print("✅ Saved as Delta table: 'employees_scd2_snapshot'")
print(f"   Total rows: {df_scd2_final.count()}")
print("   (3 original + 2 expired/new versions for changes + 1 new employee = 6 rows)")